# Check vs postw90 (full mesh)

Compares MMN and postw90 position matrices on complete source meshes. The reported percentage is a peak-normalized maximum error,

`100 * max_{k,c} |candidate_c(k) - reference_c(k)| / max_{k,c} |reference_c(k)|`,

where `c` runs over the occupied-trace `yz`, `zx`, `xy` curvature components. This avoids the singular percentages produced by pointwise normalization at zeros of the reference curvature.

Set `RUN_COMPUTATION = False` to skip straight to loading the last saved run from `results/postw90_full_mesh/`.

In [ ]:
import sys
from pathlib import Path

import json
import numpy as np
import matplotlib.pyplot as plt
from pythtb import W90

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from modules import curvature as cv
from wannier.wannier_io import read_wannier_checkpoint
from wannier.position_matrix import (
    apply_orbital_dependent_R_mapping,
    build_position_matrix_from_mmn,
    install_postw90_position_matrix,
)
from validation._paths import RESULTS, STO_TI_Q_2A_RUN
from validation._paths import DATA

PREFIX = "SrTiO3"
N_OCC = 38
COMPONENT_NAMES = ("yz", "zx", "xy")
COMPONENTS = ((1, 2), (2, 0), (0, 1))

## Config

All of these come from an external postw90 run on a complete k-mesh plus the matching MMN pair-distance cache -- there is no in-repo default, they must be pointed at real files. `OUTPUT_DIR` defaults to the existing `results/postw90_full_mesh/` (the original CLI required `--output-directory` explicitly; this default just matches what's already there).

In [ ]:
RUN_COMPUTATION = False  # True: rebuild from a real postw90 full-mesh run and recompute. False: load results/postw90_full_mesh/.
TRIAL03_DIRECTORY = STO_TI_Q_2A_RUN / "trial_03_Sr_sp_Ti_pd_O_sp/proj_gauge"
TRIAL04_DIRECTORY = DATA / "SrTiO3/Ti_Q_2A/output/188914bc889/trial_04_Sr_sp_Ti_p_O_sp/proj_gauge/188927bc889"
TRIAL03_CHECKPOINT = TRIAL03_DIRECTORY / f"{PREFIX}.chk"
TRIAL04_CHECKPOINT = TRIAL04_DIRECTORY / f"{PREFIX}.chk"
TRIAL03_POSTW90_POSITION = None  # Path to trial03's postw90 _r_full.dat on the full mesh
TRIAL04_POSTW90_POSITION = None  # Path to trial04's postw90 _r_full.dat on the full mesh
TRIAL03_MMN_CACHE = None  # Path to trial03's _AA_cache npz for this mesh
TRIAL04_MMN_CACHE = None  # Path to trial04's _AA_cache npz for this mesh
KPOINTS_NPZ = None  # Path to an npz with a "source_kpoints_reduced" array
MESH_LABEL = "8^3"
OUTPUT_DIR = RESULTS / "postw90_full_mesh"

## Helpers

In [ ]:
def repository_trace(model, kpoints: np.ndarray, chunk_size: int = 64) -> np.ndarray:
    """Evaluate the occupied trace in chunks to keep memory bounded."""
    reciprocal_inverse = np.linalg.inv(np.asarray(model.recip_lat_vecs, dtype=float))
    pos = cv.position_terms(model, dtype=np.complex128)
    result = []
    for start in range(0, len(kpoints), chunk_size):
        points = kpoints[start : start + chunk_size]
        curvature = cv.omega(cv.connection(cv.fields(model, pos, points), N_OCC))
        curvature = np.einsum("ia,jb,ab...->ij...", reciprocal_inverse, reciprocal_inverse, curvature)
        result.append(np.column_stack([np.trace(curvature[i, j], axis1=-2, axis2=-1).real for i, j in COMPONENTS]))
    return np.concatenate(result)


def mapped_model(directory: Path, checkpoint_path: Path):
    checkpoint = read_wannier_checkpoint(checkpoint_path)
    w90 = W90(str(directory), PREFIX)
    apply_orbital_dependent_R_mapping(w90, checkpoint["mp_grid"], verbose=False)
    return w90, w90.model(zero_energy=0.0, min_hopping_norm=0.0), checkpoint


def peak_normalized_max(reference: np.ndarray, candidate: np.ndarray, kpoints: np.ndarray) -> dict:
    difference = np.abs(candidate - reference)
    flat_index = int(np.argmax(difference))
    point_index, component_index = np.unravel_index(flat_index, difference.shape)
    reference_peak = float(np.max(np.abs(reference)))
    maximum_error = float(difference[point_index, component_index])
    return {
        "definition": "100 * max_{k,c}|candidate-reference| / max_{k,c}|reference|",
        "max_percent_difference": 100.0 * maximum_error / reference_peak,
        "max_abs_difference_A2": maximum_error,
        "reference_peak_abs_A2": reference_peak,
        "maximum_location": {
            "k_reduced": kpoints[point_index].tolist(),
            "component": COMPONENT_NAMES[component_index],
            "reference_A2": float(reference[point_index, component_index]),
            "candidate_A2": float(candidate[point_index, component_index]),
        },
    }


def evaluate_case(directory, checkpoint_path, postw90_position_path, mmn_cache_path, kpoints):
    w90, post_model, checkpoint = mapped_model(directory, checkpoint_path)
    centers_cartesian = np.asarray(w90.lattice.orb_vecs, dtype=float) @ np.asarray(w90.lat, dtype=float)
    install_postw90_position_matrix(post_model, postw90_position_path, centers_cartesian=centers_cartesian)
    post_trace = repository_trace(post_model, kpoints)

    _, mmn_model, _ = mapped_model(directory, checkpoint_path)
    build_position_matrix_from_mmn(
        mmn_model, directory, PREFIX, mmn_directory=directory, cache=mmn_cache_path,
        translational_invariant_JM=False, translational_invariant_MV=False, R_distance_tolerance=1e-5,
    )
    mmn_trace = repository_trace(mmn_model, kpoints)
    return post_trace, mmn_trace

## Compute both trials' postw90-installed and MMN-derived traces

`traces["trial03_postw90_A"]` etc. are the raw curvature-trace arrays, directly inspectable.

In [ ]:
if RUN_COMPUTATION:
    with np.load(KPOINTS_NPZ) as stored:
        kpoints = np.asarray(stored["source_kpoints_reduced"], float)

    directories = {"trial03": TRIAL03_DIRECTORY, "trial04": TRIAL04_DIRECTORY}
    checkpoints = {"trial03": TRIAL03_CHECKPOINT, "trial04": TRIAL04_CHECKPOINT}
    postw90_positions = {"trial03": TRIAL03_POSTW90_POSITION, "trial04": TRIAL04_POSTW90_POSITION}
    mmn_caches = {"trial03": TRIAL03_MMN_CACHE, "trial04": TRIAL04_MMN_CACHE}

    traces = {}
    for trial in ("trial03", "trial04"):
        traces[f"{trial}_postw90_A"], traces[f"{trial}_mmn_A"] = evaluate_case(
            directories[trial], checkpoints[trial], postw90_positions[trial], mmn_caches[trial], kpoints,
        )

## Summary: position-construction and trial-set peak-normalized errors

In [ ]:
if RUN_COMPUTATION:
    summary = {
        "mesh": MESH_LABEL,
        "n_kpoints": len(kpoints),
        "metric": "peak-normalized maximum absolute difference",
        "position_construction": {
            trial: peak_normalized_max(traces[f"{trial}_postw90_A"], traces[f"{trial}_mmn_A"], kpoints)
            for trial in ("trial03", "trial04")
        },
        "trial_set": {
            construction: peak_normalized_max(
                traces[f"trial03_{construction}"], traces[f"trial04_{construction}"], kpoints,
            )
            for construction in ("postw90_A", "mmn_A")
        },
    }
    print(json.dumps(summary, indent=2))

## Save

In [ ]:
if RUN_COMPUTATION:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_path = OUTPUT_DIR / "full_mesh_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2) + "\n")
    np.savez_compressed(OUTPUT_DIR / "full_mesh_curvature.npz", kpoints_reduced=kpoints, **traces)
    print(f"saved {summary_path}")
    print(f"saved {OUTPUT_DIR / 'full_mesh_curvature.npz'}")

## Load previously saved results (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    summary = json.loads((OUTPUT_DIR / "full_mesh_summary.json").read_text())
    with np.load(OUTPUT_DIR / "full_mesh_curvature.npz") as stored:
        kpoints = stored["kpoints_reduced"]
        traces = {key: stored[key] for key in stored.files if key != "kpoints_reduced"}
    print(json.dumps(summary, indent=2))

## Plot: postw90 vs MMN trace, both trials, xy component

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=150)
for ax, trial in zip(axes, ("trial03", "trial04")):
    post = traces[f"{trial}_postw90_A"][:, 2]
    mmn = traces[f"{trial}_mmn_A"][:, 2]
    order = np.argsort(post)
    ax.plot(post[order], label="postw90", linewidth=1.5)
    ax.plot(mmn[order], label="mmn", linewidth=1, linestyle="--")
    ax.set_title(f"{trial}: xy component, points sorted by postw90 value")
    ax.set_ylabel(r"$\mathrm{Tr}\,\Omega_{xy}$ [$\AA^2$]")
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()